In [ ]:
"""
INTELLI-CREDIT | ML Layer — Notebook 5: Pre-Cognitive Analysis
==============================================================
Run AFTER notebooks 1–4. CPU fine. ~5 minutes.

WHAT THIS DOES:
  Generates forward-looking risk signals that traditional credit analysis misses.
  Four components:
    1. Velocity Trends       — month-over-month acceleration/deceleration
    2. Triangulation Flags   — cross-source revenue/debt mismatches
    3. Sentiment Divergence  — qualitative vs quantitative contradictions
    4. Network Contagion     — graph-derived propagation risk

OUTPUT:
  precognitive_signals.csv  ← fills the missing rulebook table exactly
  precognitive_summary.csv  ← one row per company, rollup for Five Cs

SCHEMA (matches rulebook):
  signal_id, case_id, signal_category, signal_description,
  severity, data_sources_used, feeds_into_c, score_impact
"""

# !pip install -q pandas numpy

import uuid
import json
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

BASE_PATH   = Path("/content/drive/Othercomputers/My PC/data_u")
ML_OUT      = Path("/content/drive/MyDrive/IntelliSense/ml_outputs")
OUTPUT_PATH = ML_OUT
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# ── Load all source data ───────────────────────────────────────────────────────
print("⏳ Loading data...")

fin_df      = pd.read_csv(BASE_PATH / "structured" / "companies_financial_scenarios.csv")
gst_df      = pd.read_csv(BASE_PATH / "structured" / "gst_filings.csv")
itr_df      = pd.read_csv(BASE_PATH / "structured" / "itr_financials.csv")
bank_df     = pd.read_csv(BASE_PATH / "structured" / "bank_monthly_summary.csv")
borrow_df   = pd.read_csv(BASE_PATH / "structured" / "borrowing_profile_synthetic.csv")
pledge_df   = pd.read_csv(BASE_PATH / "unstructured" / "shareholding_pattern" / "promoter_pledge_analysis.csv")
mgmt_df     = pd.read_csv(BASE_PATH / "primary insights" / "management_interview_cleaned.csv")
port_df     = pd.read_csv(BASE_PATH / "structured" / "portfolio_performance.csv")

# ML outputs
graph_risk  = pd.read_csv(ML_OUT / "graph_risk_scores.csv")         if (ML_OUT / "graph_risk_scores.csv").exists()        else pd.DataFrame()
finbert_sum = pd.read_csv(ML_OUT / "finbert_summary.csv")           if (ML_OUT / "finbert_summary.csv").exists()          else pd.DataFrame()
news_sigs   = pd.read_csv(ML_OUT / "news_risk_signals.csv")         if (ML_OUT / "news_risk_signals.csv").exists()        else pd.DataFrame()
five_cs     = pd.read_csv(ML_OUT / "five_cs_scores.csv")            if (ML_OUT / "five_cs_scores.csv").exists()           else pd.DataFrame()

print(f"  fin_df     : {len(fin_df):,} companies")
print(f"  gst_df     : {len(gst_df):,} rows")
print(f"  itr_df     : {len(itr_df):,} rows")
print(f"  bank_df    : {len(bank_df):,} rows")
print(f"  borrow_df  : {len(borrow_df):,} rows")
print(f"  mgmt_df    : {len(mgmt_df):,} rows")
print(f"  port_df    : {len(port_df):,} rows")
print("✅ Loaded")

# ── Helpers ────────────────────────────────────────────────────────────────────
def make_signal(case_id, category, description, severity, sources, feeds_into_c, score_impact):
    return {
        "signal_id"          : str(uuid.uuid4()),
        "case_id"            : case_id,
        "signal_category"    : category,
        "signal_description" : description,
        "severity"           : severity,
        "data_sources_used"  : json.dumps(sources),
        "feeds_into_c"       : feeds_into_c,
        "score_impact"       : round(float(score_impact), 2),
        "generated_timestamp": datetime.utcnow().isoformat(),
    }

def safe(val, default=0.0):
    try:
        v = float(val)
        return default if np.isnan(v) else v
    except:
        return default

all_signals = []

# ── Get a unique list of companies to iterate over ────────────────────────────
companies = fin_df[["case_id", "company_id", "company_cin", "sector"]].drop_duplicates()
print(f"\n⏳ Running pre-cognitive analysis on {len(companies)} companies...")


# ============================================================
# COMPONENT 1: VELOCITY TRENDS
# ============================================================
"""
Month-over-month acceleration / deceleration in:
  - Revenue (from bank monthly credits as proxy)
  - Debt (from pledge trend)
  - NPA (from portfolio performance, NBFCs only)
"""

# ── Revenue velocity from bank monthly credits ─────────────────────────────────
if "company_id" in bank_df.columns and "total_credits" in bank_df.columns:

    for _, row in companies.iterrows():
        cid     = row["company_id"]
        case_id = row["case_id"]

        co_bank = bank_df[bank_df["company_id"] == cid].copy()
        if len(co_bank) < 4:
            continue

        # Sort by month
        if "month" in co_bank.columns:
            co_bank = co_bank.sort_values("month")

        credits = co_bank["total_credits"].dropna().values
        if len(credits) < 4:
            continue

        # Compare last 3 months avg vs prior 3 months avg
        recent   = np.mean(credits[-3:])
        prior    = np.mean(credits[-6:-3]) if len(credits) >= 6 else np.mean(credits[:3])

        if prior == 0:
            continue

        pct_change = (recent - prior) / prior * 100

        if pct_change <= -20:
            severity = "critical" if pct_change <= -35 else "high"
            impact   = -1.5 if pct_change <= -35 else -1.0
            desc = (
                f"Bank credit inflows have declined {abs(pct_change):.1f}% in the last 3 months "
                f"compared to the prior period (recent avg ₹{recent/1e7:.1f}Cr vs prior ₹{prior/1e7:.1f}Cr). "
                f"Indicates revenue deceleration not yet visible in annual financials."
            )
            all_signals.append(make_signal(
                case_id, "velocity_trend", desc, severity,
                ["Bank Monthly Summary"], "Capacity", impact
            ))
        elif pct_change >= 25:
            desc = (
                f"Bank credit inflows have grown {pct_change:.1f}% in the last 3 months "
                f"compared to the prior period. Positive revenue momentum."
            )
            all_signals.append(make_signal(
                case_id, "velocity_trend", desc, "low",
                ["Bank Monthly Summary"], "Capacity", +0.5
            ))

# ── Debt velocity from promoter pledge trend ──────────────────────────────────
if "company_id" in pledge_df.columns:

    for _, row in companies.iterrows():
        cid     = row["company_id"]
        case_id = row["case_id"]

        co_pledge = pledge_df[pledge_df["company_id"] == cid].copy()
        if co_pledge.empty:
            continue

        # Check trend column
        trend = str(co_pledge.iloc[-1].get("trend", "stable")).lower() if "trend" in co_pledge.columns else "stable"
        qoq   = safe(co_pledge.iloc[-1].get("qoq_change_percentage_points", 0))

        if trend == "increasing" and qoq > 5:
            severity = "critical" if qoq > 15 else "high"
            impact   = -1.5 if qoq > 15 else -0.8
            desc = (
                f"Promoter pledge has increased by {qoq:.1f} percentage points quarter-on-quarter "
                f"and is on an increasing trend. Rising pledge indicates promoter may be facing "
                f"personal liquidity stress and borrowing against shares."
            )
            all_signals.append(make_signal(
                case_id, "velocity_trend", desc, severity,
                ["Promoter Pledge Analysis"], "Character", impact
            ))

# ── NPA velocity (NBFCs only) ─────────────────────────────────────────────────
NBFC_SECTORS = {"NBFC", "Financial Services"}

if not port_df.empty and "company_id" in port_df.columns:

    for _, row in companies.iterrows():
        cid     = row["company_id"]
        case_id = row["case_id"]

        if row["sector"] not in NBFC_SECTORS:
            continue

        co_port = port_df[port_df["company_id"] == cid].copy()
        if len(co_port) < 2:
            continue

        npa_vals = co_port["gross_npa_pct"].dropna().values if "gross_npa_pct" in co_port.columns else []
        if len(npa_vals) < 2:
            continue

        npa_change = npa_vals[-1] - npa_vals[0]

        if npa_change > 2:
            severity = "critical" if npa_change > 5 else "high"
            impact   = -1.5 if npa_change > 5 else -1.0
            desc = (
                f"Gross NPA has increased by {npa_change:.2f} percentage points over the observed period "
                f"(from {npa_vals[0]:.2f}% to {npa_vals[-1]:.2f}%). "
                f"Rising NPA velocity signals deteriorating asset quality in the loan portfolio."
            )
            all_signals.append(make_signal(
                case_id, "velocity_trend", desc, severity,
                ["Portfolio Performance"], "Capacity", impact
            ))

print(f"  Component 1 (Velocity): {len(all_signals)} signals so far")


# ============================================================
# COMPONENT 2: TRIANGULATION FLAGS
# ============================================================
"""
Cross-verification across data sources:
  A. GST revenue vs Financial Statement revenue
  B. Borrowing profile debt vs Balance Sheet debt
  C. ITR income vs Financial Statement income (if available)
"""

# ── A. GST vs Financial Statement revenue ─────────────────────────────────────
if "company_id" in gst_df.columns and "gstr3b_revenue_declared" in gst_df.columns:

    for _, row in companies.iterrows():
        cid     = row["company_id"]
        case_id = row["case_id"]

        co_gst = gst_df[gst_df["company_id"] == cid]
        if co_gst.empty:
            continue

        # Annual GST revenue = sum of monthly declarations
        gst_annual_revenue = safe(co_gst["gstr3b_revenue_declared"].sum()) * 12 / max(1, len(co_gst))

        # Financial statement revenue
        fin_row     = fin_df[fin_df["company_id"] == cid]
        if fin_row.empty:
            continue
        fs_revenue  = safe(fin_row.iloc[0].get("revenue_cr", 0)) * 1e7  # convert Cr to absolute

        if fs_revenue == 0 or gst_annual_revenue == 0:
            continue

        divergence_pct = abs(fs_revenue - gst_annual_revenue) / fs_revenue * 100

        if divergence_pct > 20:
            direction = "higher" if fs_revenue > gst_annual_revenue else "lower"
            severity  = "high" if divergence_pct > 30 else "medium"
            impact    = -1.2 if divergence_pct > 30 else -0.8

            fs_cr  = fs_revenue / 1e7
            gst_cr = gst_annual_revenue / 1e7

            desc = (
                f"Financial Statement revenue (₹{fs_cr:.1f}Cr) is {divergence_pct:.1f}% {direction} than "
                f"annualised GST-declared revenue (₹{gst_cr:.1f}Cr). "
                f"Divergence above 20% suggests {'potential revenue inflation in financial statements' if direction == 'higher' else 'possible GST under-declaration'}."
            )
            all_signals.append(make_signal(
                case_id, "triangulation_flag", desc, severity,
                ["GST Filings (GSTR-3B)", "Financial Statements"], "Character", impact
            ))

# ── B. Borrowing profile debt vs Balance Sheet debt ───────────────────────────
if not borrow_df.empty and "company_id" in borrow_df.columns:

    # Find debt amount column
    debt_col = next(
        (c for c in borrow_df.columns if any(k in c.lower() for k in
         ["outstanding", "outstanding_amount", "debt_outstanding"])), None
    )

    if debt_col:
        for _, row in companies.iterrows():
            cid     = row["company_id"]
            case_id = row["case_id"]

            co_borrow = borrow_df[borrow_df["company_id"] == cid]
            if co_borrow.empty:
                continue

            borrow_total = safe(co_borrow[debt_col].sum())

            fin_row  = fin_df[fin_df["company_id"] == cid]
            if fin_row.empty:
                continue
            bs_debt  = safe(fin_row.iloc[0].get("debt_cr", 0)) * 1e7

            if bs_debt == 0 or borrow_total == 0:
                continue

            divergence_pct = abs(bs_debt - borrow_total) / bs_debt * 100

            if divergence_pct > 15:
                direction = "higher" if borrow_total > bs_debt else "lower"
                severity  = "high" if divergence_pct > 25 else "medium"
                impact    = -1.0 if divergence_pct > 25 else -0.6

                bs_cr = bs_debt / 1e7
                bp_cr = borrow_total / 1e7

                desc = (
                    f"Borrowing profile shows total outstanding debt of ₹{bp_cr:.1f}Cr, which is "
                    f"{divergence_pct:.1f}% {direction} than Balance Sheet debt of ₹{bs_cr:.1f}Cr. "
                    f"{'Undisclosed debt likely — off-balance-sheet exposure.' if direction == 'higher' else 'Possible borrowing profile is incomplete.'}"
                )
                all_signals.append(make_signal(
                    case_id, "triangulation_flag", desc, severity,
                    ["Borrowing Profile", "Balance Sheet"], "Capital", impact
                ))

# ── C. ITR vs Financial Statement income ──────────────────────────────────────
if not itr_df.empty and "company_id" in itr_df.columns:

    itr_income_col = next(
        (c for c in itr_df.columns if any(k in c.lower() for k in
         ["declared_gross_income", "gross_income", "net_income"])), None
    )

    if itr_income_col:
        for _, row in companies.iterrows():
            cid     = row["company_id"]
            case_id = row["case_id"]

            co_itr = itr_df[itr_df["company_id"] == cid]
            if co_itr.empty:
                continue

            itr_income = safe(co_itr.iloc[-1].get(itr_income_col, 0))

            fin_row = fin_df[fin_df["company_id"] == cid]
            if fin_row.empty:
                continue
            fs_profit = safe(fin_row.iloc[0].get("profit_cr", 0)) * 1e7

            if fs_profit == 0 or itr_income == 0:
                continue

            divergence_pct = abs(fs_profit - itr_income) / max(fs_profit, itr_income) * 100

            if divergence_pct > 10:
                itr_cr = itr_income / 1e7
                fs_cr  = fs_profit / 1e7
                direction = "lower" if itr_income < fs_profit else "higher"
                severity  = "high" if divergence_pct > 20 else "medium"
                impact    = -1.0 if divergence_pct > 20 else -0.8

                desc = (
                    f"Income declared in ITR (₹{itr_cr:.1f}Cr) is {divergence_pct:.1f}% {direction} than "
                    f"Financial Statement profit (₹{fs_cr:.1f}Cr). "
                    f"Divergence above 10% suggests {'potential ITR underreporting to reduce tax liability' if direction == 'lower' else 'financial statement profit inflation'}."
                )
                all_signals.append(make_signal(
                    case_id, "triangulation_flag", desc, severity,
                    ["ITR Financials", "Financial Statements"], "Character", impact
                ))

print(f"  Component 2 (Triangulation): {len(all_signals)} signals so far")


# ============================================================
# COMPONENT 3: SENTIMENT DIVERGENCE
# ============================================================
"""
Qualitative vs quantitative contradictions:
  A. Management interview confidence vs financial performance
  B. News sentiment (negative) vs capacity score (adequate)
  C. Pledge risk (high) vs management claiming no stress
"""

CREDIBILITY_SCORE = {
    "confident_and_consistent": 1.0,
    "hesitant_but_reasonable" : 0.5,
    "evasive"                 : -0.5,
    "contradictory_to_documents": -1.0,
    "unable_to_explain"       : -0.8,
}

if not mgmt_df.empty and "company_id" in mgmt_df.columns:

    for _, row in companies.iterrows():
        cid     = row["company_id"]
        case_id = row["case_id"]

        co_mgmt = mgmt_df[mgmt_df["company_id"] == cid]
        if co_mgmt.empty:
            continue

        # Average management credibility
        if "management_credibility_assessment" not in co_mgmt.columns:
            continue

        cred_scores = co_mgmt["management_credibility_assessment"].map(
            CREDIBILITY_SCORE
        ).dropna()
        if cred_scores.empty:
            continue

        avg_cred = cred_scores.mean()

        # ── A. Confident management vs declining financials ────────────────────
        fin_row   = fin_df[fin_df["company_id"] == cid]
        if not fin_row.empty:
            profit    = safe(fin_row.iloc[0].get("profit_cr", 0))
            revenue   = safe(fin_row.iloc[0].get("revenue_cr", 1))
            pat_margin = profit / revenue * 100 if revenue > 0 else 0

            if avg_cred >= 0.8 and pat_margin < 5:
                desc = (
                    f"Management presented as confident and consistent during interviews, "
                    f"yet PAT margin stands at {pat_margin:.1f}% — below the 5% threshold indicating "
                    f"operational stress. Confident tone despite weak financials may indicate "
                    f"management is not fully disclosing challenges."
                )
                all_signals.append(make_signal(
                    case_id, "sentiment_divergence", desc, "medium",
                    ["Management Interview", "Financial Statements"], "Character", -0.8
                ))

        # ── B. Evasive management vs adequate capacity score ──────────────────
        if avg_cred <= -0.5 and not five_cs.empty:
            cs_row = five_cs[five_cs["company_id"] == cid] if "company_id" in five_cs.columns else pd.DataFrame()
            if not cs_row.empty:
                cap_score = safe(cs_row.iloc[0].get("capacity_score", 5))
                if cap_score >= 6.0:
                    desc = (
                        f"Management interview responses were evasive or contradictory to documents "
                        f"(credibility score: {avg_cred:.2f}), yet the quantitative Capacity score "
                        f"appears adequate at {cap_score:.1f}/10. This divergence warrants deeper "
                        f"due diligence — financials may not yet reflect emerging stress."
                    )
                    all_signals.append(make_signal(
                        case_id, "sentiment_divergence", desc, "high",
                        ["Management Interview", "Five Cs Scorecard"], "Character", -1.0
                    ))

        # ── C. Contradictory management vs high pledge ────────────────────────
        co_pledge = pledge_df[pledge_df["company_id"] == cid] if "company_id" in pledge_df.columns else pd.DataFrame()
        if not co_pledge.empty and avg_cred <= -0.5:
            risk_flag = str(co_pledge.iloc[-1].get("pledge_risk_flag", "normal"))
            if risk_flag in ["high_risk", "critical"]:
                pledge_pct = safe(co_pledge.iloc[-1].get("promoter_pledged_pct", 0))
                desc = (
                    f"Promoter pledge is at {pledge_pct:.1f}% ({risk_flag.replace('_', ' ')} level) "
                    f"while management interview responses were evasive or inconsistent. "
                    f"High pledge combined with poor interview credibility is a strong Character C warning signal."
                )
                all_signals.append(make_signal(
                    case_id, "sentiment_divergence", desc, "high",
                    ["Management Interview", "Promoter Pledge Analysis"], "Character", -1.2
                ))

# ── D. News sentiment negative vs stable rating ───────────────────────────────
if not news_sigs.empty and not finbert_sum.empty:

    for _, row in companies.iterrows():
        cid     = row["company_id"]
        case_id = row["case_id"]

        co_news = news_sigs[news_sigs["company_id"] == cid] if "company_id" in news_sigs.columns else pd.DataFrame()
        fb_row  = finbert_sum[finbert_sum["company_id"] == cid] if "company_id" in finbert_sum.columns else pd.DataFrame()

        if co_news.empty or fb_row.empty:
            continue

        high_sev_news  = len(co_news[co_news.get("is_high_severity", False)])
        finbert_risk   = safe(fb_row.iloc[0].get("finbert_risk_score", 0))

        # High negative news but capacity score looks fine
        if high_sev_news >= 3 and not five_cs.empty:
            cs_row = five_cs[five_cs["company_id"] == cid] if "company_id" in five_cs.columns else pd.DataFrame()
            if not cs_row.empty:
                cap_score = safe(cs_row.iloc[0].get("capacity_score", 5))
                char_score = safe(cs_row.iloc[0].get("character_score", 5))
                if cap_score >= 6 and finbert_risk > 0.6:
                    desc = (
                        f"{high_sev_news} high-severity news risk signals detected (FinBERT risk score: "
                        f"{finbert_risk:.2f}) while quantitative Capacity score is {cap_score:.1f}/10. "
                        f"Negative news sentiment typically leads financial deterioration by 1–2 quarters. "
                        f"This divergence is a leading indicator of potential stress."
                    )
                    all_signals.append(make_signal(
                        case_id, "sentiment_divergence", desc, "high",
                        ["News Articles", "FinBERT Analysis", "Five Cs Scorecard"], "Conditions", -0.8
                    ))

print(f"  Component 3 (Sentiment Divergence): {len(all_signals)} signals so far")


# ============================================================
# COMPONENT 4: NETWORK CONTAGION
# ============================================================
"""
Graph-derived propagation risk:
  A. Director's other companies have DRT/NCLT cases
  B. High graph contagion risk score from Graph Engine
  C. Sector peers are anomalous (Isolation Forest)
"""

if not graph_risk.empty:

    for _, row in companies.iterrows():
        cid     = row["company_id"]
        cin     = row["company_cin"]
        case_id = row["case_id"]
        sector  = row["sector"]

        # Match by CIN
        gr_row = graph_risk[graph_risk["company_cin"] == cin] if "company_cin" in graph_risk.columns else pd.DataFrame()
        if gr_row.empty:
            gr_row = graph_risk[graph_risk["company_id"] == cid] if "company_id" in graph_risk.columns else pd.DataFrame()
        if gr_row.empty:
            continue

        g = gr_row.iloc[0]

        # ── A. Director stress contagion ───────────────────────────────────────
        avg_dir_stress = safe(g.get("avg_director_stress_score", 0))
        n_directors    = int(safe(g.get("n_directors", 0)))

        if avg_dir_stress > 4 and n_directors > 0:
            severity = "critical" if avg_dir_stress > 7 else "high"
            impact   = -1.5 if avg_dir_stress > 7 else -1.0
            desc = (
                f"Directors of this company have an average stress score of {avg_dir_stress:.1f}/10 "
                f"based on their other directorships. This indicates that promoter-linked entities "
                f"are experiencing financial stress, which historically propagates to connected companies."
            )
            all_signals.append(make_signal(
                case_id, "network_contagion", desc, severity,
                ["MCA Directors", "Director Company Network", "Legal Cases"], "Character", -impact
            ))

        # ── B. Active DRT in connected network ────────────────────────────────
        has_drt = bool(g.get("has_active_drt_case", False))
        if has_drt:
            desc = (
                f"An active DRT (Debt Recovery Tribunal) case is associated with this company or its "
                f"director network. DRT cases indicate a lender has already classified the borrower as "
                f"a defaulter and initiated legal recovery. This is a critical Character C signal."
            )
            all_signals.append(make_signal(
                case_id, "network_contagion", desc, "critical",
                ["MCA Legal Cases", "Director Company Network"], "Character", -2.0
            ))

        # ── C. High contagion risk score ───────────────────────────────────────
        contagion = safe(g.get("contagion_risk_score", 0))
        if contagion > 0.4:
            severity = "high" if contagion > 0.6 else "medium"
            impact   = -1.0 if contagion > 0.6 else -0.5
            live_charges_cr = safe(g.get("live_charges_total_inr", 0)) / 1e7
            desc = (
                f"Network contagion risk score is {contagion:.2f} — indicating that a significant "
                f"proportion of entities connected to this borrower through shared directors show "
                f"stress signals. Live charges on assets total ₹{live_charges_cr:.1f}Cr. "
                f"Stress in connected entities can propagate through supplier/customer dependencies."
            )
            all_signals.append(make_signal(
                case_id, "network_contagion", desc, severity,
                ["Graph Risk Engine", "MCA Charges"], "Conditions", impact
            ))

print(f"  Component 4 (Network Contagion): {len(all_signals)} signals so far")


# ============================================================
# SAVE OUTPUTS
# ============================================================
signals_df = pd.DataFrame(all_signals)

# ── Fill missing columns to match rulebook schema exactly ─────────────────────
for col in ["signal_id", "case_id", "signal_category", "signal_description",
            "severity", "data_sources_used", "feeds_into_c", "score_impact"]:
    if col not in signals_df.columns:
        signals_df[col] = None

out_path = OUTPUT_PATH / "precognitive_signals.csv"
signals_df.to_csv(out_path, index=False)
print(f"\n💾 Saved: {out_path}  ({len(signals_df)} signals)")

# ── Per-company summary ────────────────────────────────────────────────────────
if not signals_df.empty:
    summary = signals_df.groupby("case_id").agg(
        total_signals          = ("signal_id", "count"),
        critical_count         = ("severity", lambda x: (x == "critical").sum()),
        high_count             = ("severity", lambda x: (x == "high").sum()),
        medium_count           = ("severity", lambda x: (x == "medium").sum()),
        total_score_impact     = ("score_impact", "sum"),
        velocity_signals       = ("signal_category", lambda x: (x == "velocity_trend").sum()),
        triangulation_signals  = ("signal_category", lambda x: (x == "triangulation_flag").sum()),
        sentiment_signals      = ("signal_category", lambda x: (x == "sentiment_divergence").sum()),
        contagion_signals      = ("signal_category", lambda x: (x == "network_contagion").sum()),
        primary_c_affected     = ("feeds_into_c", lambda x: x.mode()[0] if len(x) > 0 else "Character"),
    ).reset_index()

    # Merge company_id back in
    summary = summary.merge(
        fin_df[["case_id", "company_id", "sector"]].drop_duplicates(),
        on="case_id", how="left"
    )

    sum_path = OUTPUT_PATH / "precognitive_summary.csv"
    summary.to_csv(sum_path, index=False)
    print(f"💾 Saved: {sum_path}  ({len(summary)} companies)")

    print(f"\nSignal breakdown:")
    print(f"  Velocity trends      : {(signals_df['signal_category'] == 'velocity_trend').sum()}")
    print(f"  Triangulation flags  : {(signals_df['signal_category'] == 'triangulation_flag').sum()}")
    print(f"  Sentiment divergence : {(signals_df['signal_category'] == 'sentiment_divergence').sum()}")
    print(f"  Network contagion    : {(signals_df['signal_category'] == 'network_contagion').sum()}")
    print(f"\nSeverity breakdown:")
    print(signals_df["severity"].value_counts().to_string())


# ============================================================
# UGRO CAPITAL SPOT-CHECK
# ============================================================
print("\n" + "="*60)
print("UGRO CAPITAL PRE-COGNITIVE SPOT-CHECK")
print("="*60)

ugro = signals_df[
    signals_df["case_id"].str.upper().str.contains("UGRO", na=False) |
    signals_df["case_id"].isin(
        fin_df[fin_df["NAME OF COMPANY"].str.contains("Ugro", case=False, na=False)]["case_id"].values
    )
]

if ugro.empty:
    # Show top-risk company as demo substitute
    if not signals_df.empty and not summary.empty:
        top_case = summary.nlargest(1, "critical_count").iloc[0]["case_id"]
        ugro     = signals_df[signals_df["case_id"] == top_case]
        print(f"  Ugro not found — showing highest-risk case: {top_case}")
    else:
        print("  No signals generated — check data joins above")

if not ugro.empty:
    print(f"  Total signals  : {len(ugro)}")
    print(f"  Critical       : {(ugro['severity'] == 'critical').sum()}")
    print(f"  High           : {(ugro['severity'] == 'high').sum()}")
    print(f"  Score impact   : {ugro['score_impact'].sum():.2f}")
    print(f"\n  Signals:")
    for _, sig in ugro.iterrows():
        icon = "🔴" if sig["severity"] == "critical" else "🟠" if sig["severity"] == "high" else "🟡"
        print(f"\n  {icon} [{sig['signal_category']}] → {sig['feeds_into_c']} ({sig['score_impact']:+.2f})")
        print(f"     {sig['signal_description'][:200]}...")

print(f"\n✅ Pre-Cognitive Analysis complete.")
print(f"""
OUTPUT FILES:
  precognitive_signals.csv   ← fills missing rulebook table ✅
  precognitive_summary.csv   ← per-company rollup for Five Cs

INTEGRATION:
  Feed precognitive_summary.csv into 04_five_cs_aggregator.py
  Each signal's score_impact adjusts the relevant C score
  Section 6 of CAM uses signal_description text directly
""")

⏳ Loading data...
  fin_df     : 2,238 companies
  gst_df     : 53,712 rows
  itr_df     : 6,714 rows
  bank_df    : 53,712 rows
  borrow_df  : 7,117 rows
  mgmt_df    : 20,186 rows
  port_df    : 1,652 rows
✅ Loaded

⏳ Running pre-cognitive analysis on 2238 companies...
  Component 1 (Velocity): 1035 signals so far
  Component 2 (Triangulation): 5487 signals so far
  Component 3 (Sentiment Divergence): 5487 signals so far
  Component 4 (Network Contagion): 7725 signals so far

💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/precognitive_signals.csv  (7725 signals)
💾 Saved: /content/drive/MyDrive/IntelliSense/ml_outputs/precognitive_summary.csv  (2238 companies)

Signal breakdown:
  Velocity trends      : 1035
  Triangulation flags  : 4452
  Sentiment divergence : 0
  Network contagion    : 2238

Severity breakdown:
severity
high        4605
critical    2238
low          856
medium        26

UGRO CAPITAL PRE-COGNITIVE SPOT-CHECK
  Total signals  : 4
  Critical       : 1
  High